In [1]:
# filter dataset
!python MT-Preparation/filtering/filter.py ./en-zh.zh ./en-zh.en zh en

Dataframe shape (rows, columns): (231267, 2)
--- Rows with Empty Cells Deleted	--> Rows: 231267
--- Duplicates Deleted			--> Rows: 229646
--- Source-Copied Rows Deleted		--> Rows: 229640
--- Too Long Source/Target Deleted	--> Rows: 224753
--- HTML Removed			--> Rows: 224753
--- Rows will remain true-cased		--> Rows: 224753
--- Rows with Empty Cells Deleted	--> Rows: 224753
--- Source Saved: ./en-zh.zh-filtered.zh
--- Target Saved: ./en-zh.en-filtered.en


### Perform BERT-WSD on SoC Computer Cluster

1. **SSH to your SoC Computer Cluster**  

2. **Run `salloc` to acquire a GPU host:**  
   ```bash
   salloc -G nv

3. **Enter the host `slurm`:** 
    ```bash
    srun --pty bash

4. **Find number of CPUs/cores on the machine:**
    ```bash
    nproc --all
(Adjust batch size in `preprocess_file` function to match the number of CPUs)

5. **Run the file to perform WSD using BERT:**
    ```bash
    nice -n 400 python salient-wsd.py

## Gradient based salient + BERT 2senses wsd 

Gradient-based saliency helps identify which words in a sentence or context window most influence the disambiguation of ambiguous words.

Gradient mathematical intuition:
The gradient $ \frac{∂y}{∂x} $ (where $y$ is the predicted logit, $x_i$ is the embedding of token $i$) measures sensitivity:
- If $\frac{∂y}{∂x}$ is large, changing $x_i$ (e.g:`money`) significantly alters $y$ (the sense prediction).

- If small (e.g.`the`), the token has little impact.

Summing absolute gradients across dimensions gives a scalar importance score per token.

## how it works

Eg: `"I deposited money in the bank" `
1. compute sliding window (±2 sentences)
    ```bash
    " The river was calm. I deposited money in the bank. The loan was approved."

2.  Pick the highest scoring sense: `bank#0` (financial sense, logit 0.8).


3. Compute saliency score via gradients to get salient words : `money`, `deposited` etc.
    ```bash
    "money": High saliency (0.5) → Strong influence. 
    "deposited": High (0.4) → Supports financial context.
    "river": Moderate (0.3) → Competes but overridden.
    "the": Low (0.02) → Minimal impact.
    Salient Words: "money deposited river" (top 3).

4. Prepends top salient words to the sentence
   ```bash
   "money deposited river I deposited money in the bank."

5. Predict senses
    ```bash
    "I deposited#0 money#1 in the bank#0"

```bash
input_file = "./en-zh.en-filtered.en"
output_file = "./en-zh.en-filtered-gradient_based_salient-wsd.en"

In [ ]:
import torch
from transformers import BertTokenizer, BertForSequenceClassification
import nltk
from nltk.corpus import wordnet
from nltk.tokenize import word_tokenize
import re
import os
import numpy as np

nltk.download('punkt')
nltk.download('wordnet')

class BertWSDProcessor:
    def __init__(self, model_path='bert-base-uncased', device='cuda' if torch.cuda.is_available() else 'cpu'):
        self.device = device
        self.tokenizer = BertTokenizer.from_pretrained(model_path)
        self.model = BertForSequenceClassification.from_pretrained(model_path)
        self.model.to(device)
        self.model.eval()
    
    def identify_ambiguous_words(self, sentence):
        """Identify potentially ambiguous words in the sentence"""
        tokens = word_tokenize(sentence)
        ambiguous_words = []
        
        for i, token in enumerate(tokens):
            synsets = wordnet.synsets(token)
            if len(synsets) > 1:
                ambiguous_words.append((token, i))
                
        return ambiguous_words
    
    def compute_saliency(self, context, target_word, target_idx):
        """Compute saliency scores for tokens in the context relative to the target word"""
        # Tokenize and encode the full context
        inputs = self.tokenizer(context, return_tensors="pt", padding=True, truncation=True).to(self.device)
        input_ids = inputs["input_ids"]
        attention_mask = inputs["attention_mask"]
        
        # Get input embeddings and enable gradients
        embeddings = self.model.get_input_embeddings()(input_ids)
        embeddings.retain_grad()
        
        # Forward pass
        outputs = self.model(inputs_embeds=embeddings, attention_mask=attention_mask)
        logits = outputs.logits
        
        # Predict sense and backpropagate
        predicted_sense = logits.argmax(dim=1).item()
        logits[0, predicted_sense].backward()
        
        # Compute saliency as gradient magnitude
        saliency = embeddings.grad.abs().sum(dim=-1)[0].cpu().numpy()
        bert_tokens = self.tokenizer.convert_ids_to_tokens(input_ids[0])
        context_tokens = word_tokenize(context)
        
        # Map saliency to original tokens, aligning BERT and NLTK tokenizations
        token_saliency = {}
        word_idx = 0  # Index for context_tokens
        for i, token in enumerate(bert_tokens):
            if token in ['[CLS]', '[SEP]'] or token.startswith('##'):
                continue  # Skip special tokens and subwords
            if word_idx >= len(context_tokens):
                break  # Prevent out-of-range error
            token_saliency[context_tokens[word_idx]] = saliency[i]
            word_idx += 1
        
        return token_saliency, predicted_sense
    
    def get_salient_context(self, sentences, current_idx, window_size=2):
        """Extract salient words from a sliding window of previous and next sentences"""
        start_idx = max(0, current_idx - window_size)
        end_idx = min(len(sentences), current_idx + window_size + 1)
        window_sentences = sentences[start_idx:end_idx]
        
        # Combine window sentences into full context
        full_context = " ".join(window_sentences).strip()
        current_sentence = sentences[current_idx].strip()
        
        # Identify ambiguous words in the current sentence
        ambiguous_words = self.identify_ambiguous_words(current_sentence)
        if not ambiguous_words:
            return "", current_sentence
        
        # Compute saliency for each ambiguous word in the full context
        salient_words = set()
        for word, word_idx in ambiguous_words:
            saliency_scores, _ = self.compute_saliency(full_context, word, word_idx)
            
            # Sort by saliency and take top 5 (excluding the target word)
            sorted_saliency = sorted(saliency_scores.items(), key=lambda x: x[1], reverse=True)[:5]
            salient_words.update([token for token, _ in sorted_saliency if token != word])
        
        # Prepend salient words to the current sentence
        salient_prefix = " ".join(sorted(salient_words))
        enriched_sentence = f"{salient_prefix} {current_sentence}" if salient_prefix else current_sentence
        
        return salient_prefix, enriched_sentence
    
    def disambiguate_word(self, word, context, word_idx):
        """Disambiguate a word using the enriched context"""
        inputs = self.tokenizer(context, return_tensors="pt", padding=True, truncation=True).to(self.device)
        with torch.no_grad():
            outputs = self.model(**inputs)
        predicted_sense = outputs.logits.argmax().item()
        return f"{word}#{predicted_sense}"
    
    def process_sentence(self, sentence, sentences, current_idx):
        """Process a sentence with salient context from sliding window"""
        salient_prefix, enriched_sentence = self.get_salient_context(sentences, current_idx)
        ambiguous_words = self.identify_ambiguous_words(sentence)  # Original sentence
        
        processed_sentence = sentence
        for word, word_idx in ambiguous_words:
            # Adjust word_idx for the enriched sentence (account for salient prefix)
            prefix_tokens = word_tokenize(salient_prefix)
            adjusted_idx = word_idx + len(prefix_tokens)
            disambiguated_word = self.disambiguate_word(word, enriched_sentence, adjusted_idx)
            processed_sentence = re.sub(r'\b' + word + r'\b', disambiguated_word, processed_sentence, 1)
        
        return processed_sentence

def preprocess_file(input_file, output_file, batch_size=32, save_interval=1000):
    """Preprocess an entire file using BERT-WSD with resume support and sliding window"""
    processor = BertWSDProcessor()

    # Check how many lines are already processed
    processed_lines_count = 0
    if os.path.exists(output_file):
        with open(output_file, 'r', encoding='utf-8') as f:
            processed_lines_count = sum(1 for _ in f)
    
    print(f"Resuming from line {processed_lines_count}...")

    # Read all lines for sliding window context
    with open(input_file, 'r', encoding='utf-8') as f:
        all_lines = f.readlines()
    lines = all_lines[processed_lines_count:]  # Skip processed lines

    processed_lines = []
    
    # Process remaining lines with sliding window
    for i in range(len(lines)):
        line = lines[i].strip()
        processed_line = processor.process_sentence(line, all_lines, processed_lines_count + i)
        processed_lines.append(processed_line)
        
        print(f"Processed {processed_lines_count + i + 1}/{processed_lines_count + len(lines)} lines")

        # Save every `save_interval` lines
        if len(processed_lines) >= save_interval:
            with open(output_file, 'a', encoding='utf-8') as f:
                f.write('\n'.join(processed_lines) + '\n')
            print(f"Saved {len(processed_lines)} lines to {output_file}")
            processed_lines = []

    # Save any remaining lines
    if processed_lines:
        with open(output_file, 'a', encoding='utf-8') as f:
            f.write('\n'.join(processed_lines) + '\n')
        print(f"Final save: {len(processed_lines)} lines to {output_file}")

    print(f"Preprocessing complete. Output saved to {output_file}")

if __name__ == "__main__":
    input_file = "./en-zh.en-filtered.en"
    output_file = "./en-zh.en-filtered-gradient_based_salient-wsd.en"
    
    preprocess_file(input_file, output_file)

## TF-IDF Salient with wsd

**Process:**
1. Compute TF-IDF scores for words in the sliding window (±2 sentences).
2. Select top-scoring words (e.g., top 5 by TF-IDF) as salient.
3. Prepend these to the sentence, then run BERT for WSD.

    ```bash
    input_file = "./en-zh.en-filtered-salient.en"
    output_file = "./en-zh.en-filtered-salient-wsd.en"

In [ ]:
import torch
from transformers import BertTokenizer, BertForSequenceClassification
import nltk
from nltk.corpus import wordnet
from nltk.tokenize import word_tokenize
import pandas as pd
import re
import os

# Download necessary NLTK resources
nltk.download('punkt')
nltk.download('wordnet')

class BertWSDProcessor:
    def __init__(self, model_path='bert-base-uncased', device='cuda' if torch.cuda.is_available() else 'cpu'):
        self.device = device
        self.tokenizer = BertTokenizer.from_pretrained(model_path)
        self.model = BertForSequenceClassification.from_pretrained(model_path)
        self.model.to(device)
        self.model.eval()
    
    def identify_ambiguous_words(self, sentence):
        """Identify potentially ambiguous words in the sentence"""
        tokens = word_tokenize(sentence)
        ambiguous_words = []
        
        for token in tokens:
            # Check if the word has multiple senses in WordNet
            synsets = wordnet.synsets(token)
            if len(synsets) > 1:
                ambiguous_words.append(token)
                
        return ambiguous_words
    
    def disambiguate_word(self, word, context):
        """Use BERT-WSD to disambiguate a word in context"""
        # Format input for BERT
        inputs = self.tokenizer(context, return_tensors="pt").to(self.device)
        
        with torch.no_grad():
            outputs = self.model(**inputs)
        
        # Get predicted sense ID (implementation depends on your specific BERT-WSD model)
        predicted_sense = outputs.logits.argmax().item()
        
        # Map sense ID to WordNet sense (this mapping depends on your model)
        # For simplicity, we'll just return the sense ID in this example
        return f"{word}#{predicted_sense}"
    
    def process_sentence(self, sentence):
        """Process a sentence, disambiguating ambiguous words"""
        ambiguous_words = self.identify_ambiguous_words(sentence)
        processed_sentence = sentence
        
        for word in ambiguous_words:
            # Get the disambiguated sense
            disambiguated_word = self.disambiguate_word(word, sentence)
            
            # Replace the word with its disambiguated form
            # This simple replacement strategy might need improvement for real use cases
            processed_sentence = re.sub(r'\b' + word + r'\b', disambiguated_word, processed_sentence, 1)
            
        return processed_sentence

def preprocess_file(input_file, output_file, batch_size=32, save_interval=1000):
    """Preprocess an entire file using BERT-WSD with resume support."""
    processor = BertWSDProcessor()

    # Check how many lines are already processed
    processed_lines_count = 0
    if os.path.exists(output_file):
        with open(output_file, 'r', encoding='utf-8') as f:
            processed_lines_count = sum(1 for _ in f)  # Count existing lines
    
    print(f"Resuming from line {processed_lines_count}...")

    # Read input file and skip already processed lines
    with open(input_file, 'r', encoding='utf-8') as f:
        lines = f.readlines()[processed_lines_count:]  # Skip processed lines

    processed_lines = []
    
    # Process remaining lines
    for i in range(0, len(lines), batch_size):
        batch = lines[i:i+batch_size]
        
        for line in batch:
            processed_line = processor.process_sentence(line.strip())
            processed_lines.append(processed_line)
        
        print(f"Processed {processed_lines_count + min(i+batch_size, len(lines))}/{processed_lines_count + len(lines)} lines")

        # Save every `save_interval` lines
        if len(processed_lines) >= save_interval:
            with open(output_file, 'a', encoding='utf-8') as f:
                f.write('\n'.join(processed_lines) + '\n')
            print(f"Saved {len(processed_lines)} lines to {output_file}")
            processed_lines = []  # Clear the buffer

    # Save any remaining lines
    if processed_lines:
        with open(output_file, 'a', encoding='utf-8') as f:
            f.write('\n'.join(processed_lines) + '\n')
        print(f"Final save: {len(processed_lines)} lines to {output_file}")

    print(f"Preprocessing complete. Output saved to {output_file}")

if __name__ == "__main__":
    input_file = "./en-zh.en-filtered-salient.en"
    output_file = "./en-zh.en-filtered-salient-wsd.en"
    
    preprocess_file(input_file, output_file)

In [ ]:
# train a sentencepiece model for subwording
!python MT-Preparation/subwording/1-train_unigram.py ./en-zh.en-filtered-wsd-processed.en ./en-zh.zh-filtered-wsd.zh

In [ ]:
# subword the dataset
!python MT-Preparation/subwording/2-subword.py source.model target.model ./en-zh.en-filtered-wsd-processed.en ./en-zh.zh-filtered-wsd.zh

In [ ]:
# first 3 lines before subwording
!head -n 3 ./en-zh.en-filtered-wsd-processed.en && echo "-----" && head -n 3 ./en-zh.zh-filtered-wsd.zh

In [ ]:
# first 3 lines after subwording
!head -n 3 ./en-zh.en-filtered-wsd-processed.en.subword && echo "---" && head -n 3 ./en-zh.zh-filtered-wsd.zh.subword after

In [ ]:
# split the dataset into training set, development set, and test set
# Development and test sets should be between 100 and 500 segments (here we chose 200)
!python3 MT-Preparation/train_dev_split/train_dev_test_split.py 2000 2000 ./en-zh.en-filtered-wsd-processed.en.subword ./en-zh.zh-filtered-wsd.zh.subword

In [ ]:
!wc -l ./*.subword.*

In [ ]:
# check the first and last line from each dataset
!echo "---First line---"
!head -n 1 ./*.{train,dev,test}

!echo -e "\n---Last line---"
!tail -n 1 ./*.{train,dev,test}